# Simple todo agent

Can be run on CPU

Require OPENAI_API_KEY configured

Or change to a different provider

👉  also check out the same file on [Google Colab](https://colab.research.google.com/drive/1J75zxZmUwy_mFDkVFSDOUDpw2r3b82rm?usp=drive_link)

### Import libraries

In [ ]:
from rich.console import Console
from openai import OpenAI
from dotenv import load_dotenv
import json

In [ ]:
load_dotenv(override=True)
openai = OpenAI()
MODEL = "gpt-5.2"

In [ ]:
todos, completed = [], []

### Define functions

In [ ]:
def show(text):
  try:
    Console().print(text)
  except Exception:
    print(text)

In [ ]:
def get_todo_report():
  result = ""
  for index, todo in enumerate(todos):
    if completed[index]:
      result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
    else:
      result += f"Todo #{index + 1}: {todo}\n"
  show(result)
  return result

In [ ]:
def create_todos(descriptions: list[str]):
  todos.extend(descriptions)
  completed.extend([False] * len(descriptions))
  return get_todo_report()

In [ ]:
def mark_complete(index: int, completion_notes: str):
  if 1 <= index <= len(todos):
    completed[index - 1] = True
  else:
    return "No todo at this index."
  Console().print(completion_notes)
  return get_todo_report()

In [ ]:
# Test 1
create_todos(["Go for a run", "Learn some AI", "Do some work"])

In [ ]:
# Test 2
mark_complete(2, "done")

### Create JSON for tool calls

In [ ]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
            }
        },
        "required": ["descriptions"],
        "type": "object",
        "additionalProperties": False
    }
}

In [ ]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        "properties": {
            "index": {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
            },
            "completion_notes": {
                'description': 'Notes about how you completed the todo in tich console markup',
                'title': 'Completion Notes',
                'type': 'string'
            }
        },
        "required": ["index", "completion_notes"],
        "type": "object",
        "additionalProperties": False
    }
}

In [ ]:
tools = [{"type": "function", "function": create_todos_json},
         {"type": "function", "function": mark_complete_json}]

In [ ]:
def handle_tool_calls(tool_calls):
  results = []
  for tool_call in tool_calls:
    tool_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)
    tool = globals().get(tool_name)
    result = tool(**arguments) if tool else {}
    results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
  return results

In [ ]:
def loop(messages):
  done = False
  while not done:
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools, reasoning_effort="none")
    finish_reason = response.choices[0].finish_reason
    if finish_reason == "tool_calls":
      message = response.choices[0].message
      tool_calls = message.tool_calls
      results = handle_tool_calls(tool_calls)
      messages.append(message)
      messages.extend(results)
    else:
      done = True
  show(response.choices[0].message.content)

### Ask a question to LLM and use todo tool

In [ ]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""

user_message = """
A train leaves London at 2:00 pm travelling 60 mph.
Another train leaves Birmingham at 3:00 pm travelling 80 mph towards London.
When and where do they meet?
"""

messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [ ]:
todos, completed = [], []
loop(messages)